# DTM Fine-Tuning — 93.65% → 97%+
### Uploading your trained weights and pushing the final 3.35 percentage points

## Starting Point
| Metric | Value |
|---|---|
| Current val accuracy | **93.65%** |
| Current ground F1 | 0.9020 |
| Optimal threshold | 0.57 |
| Model | best_model.pth (3.58 MB) |

## Strategy to Close 3.35pp in ~40 Minutes

| Technique | Expected Gain | Time |
|---|---|---|
| Round 3 pseudo-labels (conf≥0.95, high purity) | +1.0–1.5pp | ~8 min |
| Hard Example Mining (train on worst 40% tiles) | +1.0–1.5pp | +15 min |
| Point Mixup augmentation | +0.5–1.0pp | free |
| Test-Time Augmentation at val (8 passes) | +0.3–0.8pp | free |
| **Total expected** | **+3.35pp → 97%** | **~40 min** |

## What You Need to Upload
Add your `dtm_outputs.zip` as a Kaggle Dataset before running:
```
kaggle.com → Datasets → New Dataset
Upload: dtm_outputs.zip
Name  : dtm-trained-weights
```
Also keep your training data dataset attached.

**Run order: Cells 1→2→3→4→5→6→7→8**


In [1]:
# Cell 1 — GPU setup
import torch, torch.nn as nn, torch.nn.functional as F
import random, time, math, gc, io, zipfile, json
import numpy as np
from pathlib import Path, PurePosixPath
from tqdm import tqdm

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected.\n'
        'Kaggle -> Settings -> Accelerator -> GPU T4 x2 -> Save\n'
        'Run -> Restart Session -> Run All Cells'
    )

N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda')

for i in range(N_GPUS):
    p  = torch.cuda.get_device_properties(i)
    sm = p.major * 10 + p.minor
    if sm < 70:
        raise RuntimeError(
            f'GPU {p.name} (sm_{sm}) needs PyTorch fix.\n'
            'Add Cell 0 from previous notebook (P100 fix) and run it first.'
        )
    _t = torch.zeros(4, device=f'cuda:{i}') + 1
    torch.cuda.synchronize(i); del _t
    print(f'GPU {i}: {p.name}  sm_{sm}  {p.total_memory/1e9:.1f} GB')

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

SESSION_START    = time.time()
SESSION_BUDGET_H = 11.0

print(f'\nGPUs       : {N_GPUS}')
print(f'Total VRAM : {sum(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPUS))/1e9:.1f} GB')
print(f'Session    : budget={SESSION_BUDGET_H}h')


GPU 0: Tesla T4  sm_75  15.6 GB
GPU 1: Tesla T4  sm_75  15.6 GB

GPUs       : 2
Total VRAM : 31.3 GB
Session    : budget=11.0h


In [2]:
# Cell 2 — Configuration
# Points at your TWO datasets:
#   1. Training tiles (same as before)
#   2. Your trained weights (dtm_outputs.zip)

from pathlib import Path

# ── Dataset paths ─────────────────────────────────────────────────────────────
TRAINING_DATASET_ROOT = '/kaggle/input/datasets/jaibhagwanchouhan/training-data-95pct'
WEIGHTS_DATASET_ROOT  = '/kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs'

def _find_root(base):
    base = Path(base)
    if (base/'train').exists(): return str(base), None
    for s in sorted(base.iterdir()):
        if s.is_dir() and (s/'train').exists(): return str(s), None
    zips = list(base.rglob('training_data.zip'))
    if zips: return None, str(zips[0])
    return str(base), None

_TDIR, _ZPATH = _find_root(TRAINING_DATASET_ROOT)
_USE_ZIP = _ZPATH is not None and _TDIR is None

# ── Find weights from previous training ───────────────────────────────────────
def _find_weight(name):
    """Search for a weight file in the uploaded dataset."""
    for root in [WEIGHTS_DATASET_ROOT, '/kaggle/input', '/kaggle/working']:
        for p in Path(root).rglob(name):
            return str(p)
    return None

BEST_MODEL_PATH = _find_weight('best_model.pth')
SWA_MODEL_PATH  = _find_weight('swa_model.pth')
THRESHOLD_PATH  = _find_weight('threshold.json')

print(f'Training data : {_TDIR or "ZIP:"+str(_ZPATH)}')
print(f'best_model    : {BEST_MODEL_PATH}')
print(f'swa_model     : {SWA_MODEL_PATH}')
print(f'threshold.json: {THRESHOLD_PATH}')

if not BEST_MODEL_PATH:
    raise FileNotFoundError(
        'best_model.pth not found!\n'
        'Upload dtm_outputs.zip as a Kaggle Dataset named: dtm-trained-weights\n'
        'Then add it to this notebook via Add Data.'
    )

# Load previous threshold for reference
if THRESHOLD_PATH:
    with open(THRESHOLD_PATH) as f:
        prev_meta = json.load(f)
    print(f'\nPrevious val accuracy : {prev_meta["val_accuracy"]*100:.2f}%')
    print(f'Previous F1           : {prev_meta["val_f1"]:.4f}')
    print(f'Previous threshold    : {prev_meta["threshold"]:.2f}')

BATCH_PER_GPU = 8
TOTAL_BATCH   = BATCH_PER_GPU * max(N_GPUS, 1)

CFG = {
    # Data
    'use_zip'         : _USE_ZIP,
    'zip_path'        : _ZPATH or '',
    'training_dir'    : _TDIR  or '',
    'feat_cache_dir'  : '/kaggle/working/feat_cache',
    'cons_label_dir'  : '/kaggle/working/consensus_labels',
    'pseudo_dir'      : '/kaggle/working/pseudo_labels',
    # Outputs
    'logs_dir'        : '/kaggle/working/logs',
    'model_save'      : '/kaggle/working/logs/best_model.pth',
    'swa_save'        : '/kaggle/working/logs/swa_model.pth',
    'ckpt'            : '/kaggle/working/logs/checkpoint.pth',
    'history'         : '/kaggle/working/logs/history.json',
    'curves'          : '/kaggle/working/logs/curves.png',
    'threshold'       : '/kaggle/working/logs/threshold.json',
    # Points
    'max_pts'         : 4096,
    'seed'            : 42,
    'batch_size'      : TOTAL_BATCH,
    # Fine-tune config
    'finetune_lr'     : 3e-4,    # low LR — model already trained
    'finetune_epochs' : 15,      # short — we only need +3.5pp
    'hem_epochs'      : 12,      # hard example mining epochs
    'hem_lr'          : 1e-4,    # even lower LR for HEM
    'hem_hard_frac'   : 0.40,    # train on worst 40% of tiles
    'pseudo_conf'     : 0.95,    # round 3: very high purity
    'swa_lr'          : 5e-5,
    # Loss
    'focal_alpha'     : 0.75,
    'focal_gamma'     : 2.5,     # slightly higher — focus harder on errors
    'label_smooth'    : 0.03,    # slightly less smoothing — model is already good
    # Optimiser
    'weight_decay'    : 1e-5,    # very low — prevent weight drift from trained params
    'grad_clip'       : 1.0,     # tighter clip — small LR means small updates
    # Schedule
    'use_amp'         : True,
    'val_every'       : 2,
    'patience'        : 10,
    'num_workers'     : 4,
    'prefetch_factor' : 2,
    # TTA at validation
    'val_tta_passes'  : 8,       # 8 forward passes at val time, averaged
}

for d in ['logs_dir','feat_cache_dir','cons_label_dir','pseudo_dir']:
    Path(CFG[d]).mkdir(parents=True, exist_ok=True)
for split in ['train','val']:
    (Path(CFG['cons_label_dir'])/split).mkdir(parents=True, exist_ok=True)

print(f'\nBatch size   : {TOTAL_BATCH} ({BATCH_PER_GPU}/GPU x {N_GPUS})')
print(f'Finetune LR  : {CFG["finetune_lr"]} (10x lower than original training)')
print(f'Pseudo conf  : {CFG["pseudo_conf"]} (was 0.92 in Round 2)')
print(f'HEM frac     : {CFG["hem_hard_frac"]*100:.0f}% hardest tiles')
print(f'Val TTA      : {CFG["val_tta_passes"]} passes')


Training data : /kaggle/input/datasets/jaibhagwanchouhan/training-data-95pct
best_model    : /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/best_model.pth
swa_model     : /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/swa_model.pth
threshold.json: /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/threshold.json

Previous val accuracy : 93.65%
Previous F1           : 0.9020
Previous threshold    : 0.57

Batch size   : 16 (8/GPU x 2)
Finetune LR  : 0.0003 (10x lower than original training)
Pseudo conf  : 0.95 (was 0.92 in Round 2)
HEM frac     : 40% hardest tiles
Val TTA      : 8 passes


In [3]:
# Cell 3 — Feature engineering (identical to v6 training)
# MUST be identical or loaded features will be incompatible with saved model

def _grid(xyz, cell_m, gmax=64):
    xmin,ymin=xyz[:,0].min(),xyz[:,1].min()
    xr=xyz[:,0].max()-xmin+1e-6; yr=xyz[:,1].max()-ymin+1e-6
    GW=int(np.clip(xr/cell_m,8,gmax)); GH=int(np.clip(yr/cell_m,8,gmax))
    xi=np.clip(((xyz[:,0]-xmin)/xr*GW).astype(np.int32),0,GW-1)
    yi=np.clip(((xyz[:,1]-ymin)/yr*GH).astype(np.int32),0,GH-1)
    ci=yi*GW+xi; NC=GW*GH; z=xyz[:,2].astype(np.float32)
    c_min=np.full(NC,np.inf,np.float32); c_sum=np.zeros(NC,np.float32)
    c_sq=np.zeros(NC,np.float32); c_cnt=np.zeros(NC,np.float32)
    c_max=np.full(NC,-np.inf,np.float32)
    np.minimum.at(c_min,ci,z); np.maximum.at(c_max,ci,z)
    np.add.at(c_sum,ci,z); np.add.at(c_sq,ci,z*z); np.add.at(c_cnt,ci,1.)
    empty=c_cnt==0
    c_cnt[empty]=1; c_min[empty]=z.mean(); c_max[empty]=z.mean()
    c_sum[empty]=z.mean(); c_sq[empty]=z.mean()**2
    c_mean=c_sum/c_cnt
    c_std=np.sqrt(np.maximum(c_sq/c_cnt-c_mean**2,0.))
    return ci,c_min,c_std,c_cnt,c_max-c_min,GW,GH


def geo_features(xyz: np.ndarray) -> np.ndarray:
    """(N,3) -> (N,6): dZ_2m, roughness, slope, density, dZ_8m, planarity"""
    xyz=xyz.astype(np.float32,copy=False)
    ci2,cm2,cs2,cc2,cr2,GW2,GH2=_grid(xyz,2.0)
    dz2=np.clip(xyz[:,2]-cm2[ci2],0.,None)
    roughness=cs2[ci2]
    cg=cm2.reshape(GH2,GW2); pad=np.pad(cg,1,mode='edge')
    sg=np.max(np.stack([np.abs(cg-pad[:-2,1:-1]),np.abs(cg-pad[2:,1:-1]),
                        np.abs(cg-pad[1:-1,:-2]),np.abs(cg-pad[1:-1,2:])]),axis=0)/2.
    slope=sg.reshape(-1)[ci2]
    density=cc2[ci2]/(cc2.max()+1e-6)
    ci8,cm8,_,_,_,_,_=_grid(xyz,8.0,32)
    dz8=np.clip(xyz[:,2]-cm8[ci8],0.,None)
    planarity=cs2[ci2]/(cr2[ci2]+1e-6)
    feat=np.stack([dz2,roughness,slope,density,dz8,planarity],axis=1)
    mu=feat.mean(0,keepdims=True); sg=feat.std(0,keepdims=True)+1e-6
    return ((feat-mu)/sg).astype(np.float32)


def build_feat_cache(cfg, split):
    cd=Path(cfg['feat_cache_dir'])/split; cd.mkdir(parents=True,exist_ok=True)
    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tiles={}
        for n in names:
            parts=PurePosixPath(n).parts
            if split in parts and n.endswith('points.npy'):
                tiles[parts[parts.index(split)+1]]=n
        zf_h=zipfile.ZipFile(cfg['zip_path'])
        try:
            for t,zp in tqdm(tiles.items(),desc=f'FeatCache[{split}]',unit='tile'):
                out=cd/f'{t}.npy'
                if out.exists(): continue
                with zf_h.open(zp) as fh: xyz=np.load(io.BytesIO(fh.read()))
                np.save(str(out),geo_features(xyz))
        finally: zf_h.close()
    else:
        for td in tqdm(sorted((Path(cfg['training_dir'])/split).glob('tile_*')),
                       desc=f'FeatCache[{split}]'):
            out=cd/f'{td.name}.npy'
            if not out.exists():
                np.save(str(out),geo_features(np.load(str(td/'points.npy'))))


print('Building feature caches (skips if already exist)...')
t0=time.time()
build_feat_cache(CFG,'train'); build_feat_cache(CFG,'val')
print(f'Done in {(time.time()-t0)/60:.1f} min')


Building feature caches (skips if already exist)...


FeatCache[val]: 100%|██████████| 1216/1216 [00:10<00:00, 120.03it/s]

Done in 0.9 min


In [4]:
# Cell 4 — Model definition + load trained weights
# CRITICAL: architecture must be IDENTICAL to the model that produced best_model.pth

def _fps(xyz,n):
    B,N,_=xyz.shape; dev=xyz.device
    c=torch.zeros(B,n,dtype=torch.long,device=dev)
    d=torch.full((B,N),1e10,dtype=torch.float32,device=dev)
    far=torch.randint(0,N,(B,),dtype=torch.long,device=dev)
    bi=torch.arange(B,dtype=torch.long,device=dev)
    for i in range(n):
        c[:,i]=far; cc=xyz[bi,far].unsqueeze(1)
        dd=((xyz-cc)**2).sum(-1)
        d=torch.where(dd<d,dd,d); far=d.argmax(-1)
    return c

def _idx(p,i):
    B=p.shape[0]
    bi=torch.arange(B,device=p.device).view([B]+[1]*(i.dim()-1)).expand_as(i)
    return p[bi,i]

def _ball(nxyz,xyz,r,k):
    d=torch.cdist(nxyz,xyz)
    return torch.where(d<=r,d,d.new_full((),1e10)).topk(k,dim=-1,largest=False).indices

class _MSG(nn.Module):
    def __init__(self,nc,radii,nsamps,ic,specs):
        super().__init__(); self.nc=nc; self.radii=radii; self.nsamps=nsamps
        self.mlps=nn.ModuleList()
        for dims in specs:
            ls,ch=[],ic+3
            for d in dims: ls+=[nn.Conv2d(ch,d,1,bias=False),nn.BatchNorm2d(d),nn.ReLU(True)]; ch=d
            self.mlps.append(nn.Sequential(*ls))
    def forward(self,xyz,feat):
        ci=_fps(xyz,self.nc); nxyz=_idx(xyz,ci); outs=[]
        for r,k,mlp in zip(self.radii,self.nsamps,self.mlps):
            idx=_ball(nxyz,xyz,r,k); grp=_idx(feat,idx)
            rxyz=_idx(xyz,idx)-nxyz.unsqueeze(2)
            grp=torch.cat([rxyz,grp],-1).permute(0,3,2,1)
            outs.append(mlp(grp).max(2).values)
        return nxyz,torch.cat(outs,1).permute(0,2,1)

class _SA(nn.Module):
    def __init__(self,r,k,ic,dims):
        super().__init__(); self.r=r; self.k=k
        ls,ch=[],ic+3
        for d in dims: ls+=[nn.Conv2d(ch,d,1,bias=False),nn.BatchNorm2d(d),nn.ReLU(True)]; ch=d
        self.mlp=nn.Sequential(*ls)
    def forward(self,xyz,feat):
        nc=max(1,xyz.shape[1]//4); ci=_fps(xyz,nc); nxyz=_idx(xyz,ci)
        idx=_ball(nxyz,xyz,self.r,self.k); grp=_idx(feat,idx)
        rxyz=_idx(xyz,idx)-nxyz.unsqueeze(2)
        grp=torch.cat([rxyz,grp],-1).permute(0,3,2,1)
        return nxyz,self.mlp(grp).max(2).values.permute(0,2,1)

class _FP(nn.Module):
    def __init__(self,ic,dims):
        super().__init__()
        ls,ch=[],ic
        for d in dims: ls+=[nn.Conv1d(ch,d,1,bias=False),nn.BatchNorm1d(d),nn.ReLU(True)]; ch=d
        self.mlp=nn.Sequential(*ls)
    def forward(self,xyz1,xyz2,f1,f2):
        d=torch.cdist(xyz1,xyz2); t3=d.topk(3,dim=-1,largest=False)
        w=1./(t3.values+1e-8); w=w/w.sum(-1,keepdim=True)
        interp=(_idx(f2,t3.indices)*w.unsqueeze(-1)).sum(2)
        feat=torch.cat([f1,interp],-1) if f1 is not None else interp
        return self.mlp(feat.permute(0,2,1)).permute(0,2,1)

class DTMPointNet2(nn.Module):
    """MUST match v6 architecture exactly for weight loading to work."""
    def __init__(self,in_feat=9):
        super().__init__()
        C=in_feat
        self.sa1=_MSG(512,[0.5,1.5],[32,64],C,[[64,64],[64,128]])
        self.sa2=_MSG(128,[3.0,8.0],[64,128],192,[[128,192],[128,256]])
        self.sa3=_SA(15.0,128,448,[256,512])
        self.fp3=_FP(512+448,[256,256])
        self.fp2=_FP(256+192,[256,128])
        self.fp1=_FP(128+C,[128,128,64])
        self.head=nn.Sequential(
            nn.Conv1d(64,64,1,bias=False),nn.BatchNorm1d(64),nn.ReLU(True),
            nn.Dropout(0.4),nn.Conv1d(64,2,1))
    def forward(self,pts):
        xyz0=pts[:,:,:3].contiguous(); f0=pts.contiguous()
        xyz1,f1=self.sa1(xyz0,f0); xyz2,f2=self.sa2(xyz1,f1)
        xyz3,f3=self.sa3(xyz2,f2)
        f2=self.fp3(xyz2,xyz3,f2,f3); f1=self.fp2(xyz1,xyz2,f1,f2)
        f0=self.fp1(xyz0,xyz1,f0,f1)
        return self.head(f0.permute(0,2,1)).permute(0,2,1)


IN_FEAT = 9
_base   = DTMPointNet2(IN_FEAT).to(DEVICE)

# ── Load trained weights ──────────────────────────────────────────────────────
# Try SWA first (generally better generalisation), fall back to best_epoch
if SWA_MODEL_PATH and Path(SWA_MODEL_PATH).exists():
    _base.load_state_dict(torch.load(SWA_MODEL_PATH, map_location=DEVICE))
    print(f'Loaded SWA weights: {SWA_MODEL_PATH}')
    LOADED_FROM = 'swa'
else:
    _base.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    print(f'Loaded best_epoch weights: {BEST_MODEL_PATH}')
    LOADED_FROM = 'best_epoch'

model = nn.DataParallel(_base) if N_GPUS > 1 else _base

# Sanity check — make sure architecture matches weights
_x = torch.randn(2, 64, IN_FEAT, device=DEVICE)
with torch.no_grad(): _o = model(_x)
assert _o.shape == (2, 64, 2), f'Shape mismatch: {_o.shape}'
del _x, _o; torch.cuda.empty_cache()

n_p = sum(p.numel() for p in _base.parameters())
print(f'Model: {n_p/1e6:.2f}M params | loaded from: {LOADED_FROM}')
print(f'Starting from 93.65% val accuracy — target: 97%+')


Loaded SWA weights: /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/swa_model.pth
Model: 0.88M params | loaded from: swa
Starting from 93.65% val accuracy — target: 97%+


In [5]:
# Cell 5 — Dataset with Mixup augmentation
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


class GroundDataset(Dataset):
    """
    Same as v6 but adds Point Mixup augmentation.

    Mixup: given two tiles A and B, produce a blended tile:
      pts   = lam*A_pts + (1-lam)*B_pts   (lam ~ Beta(0.4, 0.4))
      labels = round(lam*A_lbl + (1-lam)*B_lbl)
    This forces the model to learn smooth decision boundaries rather
    than memorising exact point configurations.
    """
    def __init__(self, cfg, split='train', augment=False,
                 max_tiles=None, pseudo_dir=None, mixup=False,
                 tile_subset=None):
        self.augment   = augment
        self.max_pts   = cfg['max_pts']
        self.seed      = cfg['seed']
        self.split     = split
        self.use_zip   = cfg['use_zip']
        self.mixup     = mixup and split == 'train'
        self.feat_cache= Path(cfg['feat_cache_dir'])/split
        self.pseudo_dir= Path(pseudo_dir) if pseudo_dir else None
        self.cons_dir  = Path(cfg['cons_label_dir'])/split

        if self.use_zip:
            self._zpath = cfg['zip_path']
            self._tiles = self._disc()
        else:
            self._td = Path(cfg['training_dir'])/split
            self._tiles = sorted(
                p.name for p in self._td.glob('tile_*')
                if (p/'labels.npy').exists())

        # Apply optional subset (for Hard Example Mining)
        if tile_subset is not None:
            self._tiles = [t for t in self._tiles if t in set(tile_subset)]

        if max_tiles and len(self._tiles) > max_tiles:
            rng  = np.random.default_rng(self.seed)
            pick = rng.choice(len(self._tiles), max_tiles, replace=False)
            self._tiles = [self._tiles[i] for i in sorted(pick)]

        src = 'pseudo' if pseudo_dir else 'consensus'
        print(f'  [{split}] {len(self._tiles)} tiles '
              f'| lbl={src} | mixup={self.mixup}')

    def _disc(self):
        with zipfile.ZipFile(self._zpath) as zf: names=zf.namelist()
        t=set()
        for n in names:
            p=PurePosixPath(n).parts
            if self.split in p and 'tile_' in p[-2]:
                t.add(p[p.index(self.split)+1])
        return sorted(t)

    def _load_zip(self,tile,fn):
        if not hasattr(self,'_zf') or self._zf is None:
            self._zf = zipfile.ZipFile(self._zpath)
        with self._zf.open(f'{self.split}/{tile}/{fn}') as fh:
            return np.load(io.BytesIO(fh.read()))

    def _get_labels(self,tile):
        if self.pseudo_dir is not None:
            p = self.pseudo_dir/f'{tile}.npy'
            if p.exists(): return np.load(str(p))
        p = self.cons_dir/f'{tile}.npy'
        if p.exists(): return np.load(str(p))
        if self.use_zip: return self._load_zip(tile,'labels.npy')
        return np.load(str(self._td/tile/'labels.npy'))

    def _load_tile(self, idx):
        tile = self._tiles[idx]
        rng  = np.random.default_rng(idx + self.seed*10000)
        xyz  = (self._load_zip(tile,'points.npy') if self.use_zip
                else np.load(str(self._td/tile/'points.npy')))
        lbl  = self._get_labels(tile).astype(np.float32)

        if self.augment:
            t = rng.uniform(0, 2*np.pi)
            R = np.array([[np.cos(t),-np.sin(t)],[np.sin(t),np.cos(t)]],np.float32)
            xyz[:,:2] = xyz[:,:2]@R.T
            xyz[:,2] += rng.normal(0,0.02,len(xyz)).astype(np.float32)
            xyz *= rng.uniform(0.93,1.07)
            if rng.random()>.5: xyz[:,0]*=-1
            if rng.random()>.5: xyz[:,1]*=-1
            xyz[:,0]*=rng.uniform(0.88,1.12); xyz[:,1]*=rng.uniform(0.88,1.12)
            xyz[:,2]*=rng.uniform(0.88,1.12)
            keep=rng.random(len(xyz))>.10
            if keep.sum()>64: xyz=xyz[keep]; lbl=lbl[keep]

        N = len(xyz)
        if N > self.max_pts:
            ch=rng.choice(N,self.max_pts,replace=False)
            xyz=xyz[ch]; lbl=lbl[ch]
        elif N < self.max_pts:
            pad=rng.choice(N,self.max_pts-N,replace=True)
            xyz=np.concatenate([xyz,xyz[pad]])
            lbl=np.concatenate([lbl,lbl[pad]])

        cp = self.feat_cache/f'{tile}.npy'
        f6 = (np.load(str(cp)) if cp.exists() and not self.augment
              else geo_features(xyz))
        if len(f6) != self.max_pts: f6 = geo_features(xyz)

        xn=xyz.copy(); xn[:,:2]-=xn[:,:2].mean(0)
        xn[:,2]-=float(np.median(xn[:,2])); xn/=(np.abs(xn).max()+1e-6)
        return np.concatenate([xn,f6],axis=1).astype(np.float32), lbl

    def __len__(self): return len(self._tiles)

    def __getitem__(self, idx):
        pts_a, lbl_a = self._load_tile(idx)

        if self.mixup and np.random.random() < 0.5:
            # Mix with a random second tile
            idx_b = np.random.randint(0, len(self._tiles))
            pts_b, lbl_b = self._load_tile(idx_b)
            # Beta(0.4, 0.4) tends to sample lam near 0 or 1 (not 0.5)
            # This means most mixed tiles are mostly one class — realistic
            lam = float(np.random.beta(0.4, 0.4))
            pts_a = lam * pts_a + (1 - lam) * pts_b
            lbl_a = lam * lbl_a + (1 - lam) * lbl_b

        # Convert labels: float for soft targets, int64 for hard
        # Soft labels (from mixup) need float; hard labels need int64
        lbl_hard = (lbl_a >= 0.5).astype(np.int64)
        return torch.from_numpy(pts_a), torch.from_numpy(lbl_hard)


def make_loader(cfg, split, augment=False, pseudo_dir=None,
                balanced=False, mixup=False, tile_subset=None,
                batch_size=None):
    ds = GroundDataset(cfg, split, augment=augment,
                       pseudo_dir=pseudo_dir, mixup=mixup,
                       tile_subset=tile_subset)
    bs = batch_size or cfg['batch_size']
    nw = cfg['num_workers']
    kw = dict(num_workers=nw, pin_memory=True,
              persistent_workers=(nw>0),
              prefetch_factor=cfg['prefetch_factor'] if nw>0 else None)

    if balanced and split == 'train':
        ws = []
        for tile in ds._tiles:
            lbl = ds._get_labels(tile).astype(float)
            gf  = lbl.mean(); nf = 1-gf
            ws.append(1./(min(gf,nf)+0.05))
        sampler = WeightedRandomSampler(ws, len(ws), replacement=True)
        return DataLoader(ds, batch_size=bs, sampler=sampler,
                          drop_last=True, **kw)

    shuffle = (split == 'train')
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      drop_last=shuffle, **kw)


# Smoke test
_ds = GroundDataset(CFG, 'train', max_tiles=2)
_p, _l = _ds[0]
assert _p.shape == (CFG['max_pts'], 9)
del _ds, _p, _l
print('Dataset smoke test PASSED')


  [train] 2 tiles | lbl=consensus | mixup=False
Dataset smoke test PASSED


In [6]:
# Cell 6 — Round 3: Generate high-purity pseudo-labels (conf >= 0.95)
# Single-pass, memory-safe, resumable
# conf=0.95 means only the most unambiguous points get new labels
# Expected: replaces ~8-12% of labels, all at very high accuracy

def build_consensus_cache(cfg, split):
    """Build consensus labels if not already present."""
    out_dir = Path(cfg['cons_label_dir'])/split
    out_dir.mkdir(parents=True, exist_ok=True)

    SCALES = [(1.0,0.20),(2.0,0.35),(4.0,0.60),(8.0,1.00)]

    def _consensus(xyz):
        z=xyz[:,2].astype(np.float32); votes=np.zeros(len(xyz),np.int32)
        for cm,th in SCALES:
            xmin,ymin=xyz[:,0].min(),xyz[:,1].min()
            xr=xyz[:,0].max()-xmin+1e-6; yr=xyz[:,1].max()-ymin+1e-6
            GW=int(np.clip(xr/cm,8,64)); GH=int(np.clip(yr/cm,8,64))
            xi=np.clip(((xyz[:,0]-xmin)/xr*GW).astype(np.int32),0,GW-1)
            yi=np.clip(((xyz[:,1]-ymin)/yr*GH).astype(np.int32),0,GH-1)
            ci=yi*GW+xi; c_min=np.full(GW*GH,np.inf,np.float32)
            np.minimum.at(c_min,ci,z); c_min[np.isinf(c_min)]=z.mean()
            votes+=(z-c_min[ci]<=th).astype(np.int32)
        return (votes>=3).astype(np.int8)

    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tiles={}
        for n in names:
            parts=PurePosixPath(n).parts
            if split in parts and n.endswith('points.npy'):
                tiles[parts[parts.index(split)+1]]=n
        zf_h=zipfile.ZipFile(cfg['zip_path'])
        try:
            for t,zp in tqdm(tiles.items(),desc=f'Consensus[{split}]',unit='tile'):
                out=out_dir/f'{t}.npy'
                if out.exists(): continue
                with zf_h.open(zp) as fh: xyz=np.load(io.BytesIO(fh.read()))
                np.save(str(out),_consensus(xyz))
        finally: zf_h.close()
    else:
        for td in tqdm(sorted((Path(cfg['training_dir'])/split).glob('tile_*')),
                       desc=f'Consensus[{split}]'):
            out=out_dir/f'{td.name}.npy'
            if not out.exists():
                np.save(str(out),_consensus(np.load(str(td/'points.npy'))))


# Build consensus if missing (needed as fallback for uncertain points)
n_cons_train = len(list((Path(CFG['cons_label_dir'])/'train').glob('*.npy')))
n_cons_val   = len(list((Path(CFG['cons_label_dir'])/'val').glob('*.npy')))
print(f'Consensus labels: train={n_cons_train}, val={n_cons_val}')
if n_cons_train == 0:
    print('Building consensus labels...')
    build_consensus_cache(CFG,'train'); build_consensus_cache(CFG,'val')
else:
    print('Consensus labels exist — skipping rebuild')


def generate_round3_pseudolabels(model, cfg, conf_thresh=0.95):
    """
    Single-pass pseudo-label generation.
    Memory footprint: constant ~32 MB per tile, zero accumulation.
    Fully resumable: skips tiles already written.
    """
    round_name = 'Round3'
    out_dir    = Path(cfg['pseudo_dir'])/round_name
    out_dir.mkdir(parents=True, exist_ok=True)

    base = model.module if hasattr(model,'module') else model
    base.eval()

    use_zip = cfg['use_zip']
    if use_zip:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tile_map={}
        for n in names:
            parts=PurePosixPath(n).parts
            if 'train' in parts and n.endswith('points.npy'):
                tile_map[parts[parts.index('train')+1]]=n
        tile_list=sorted(tile_map.items())
    else:
        tdir=Path(cfg['training_dir'])/'train'
        tile_list=[(p.name,str(p/'points.npy')) for p in sorted(tdir.glob('tile_*'))]

    cons_dir=Path(cfg['cons_label_dir'])/'train'
    rng=np.random.default_rng(cfg['seed'])
    max_pts=cfg['max_pts']
    n_repl=0; n_total=0

    zf_h = zipfile.ZipFile(cfg['zip_path']) if use_zip else None
    try:
        for ti,(tile,pts_path) in enumerate(tqdm(
                tile_list,desc=f'Round3 pseudo-labels (conf>={conf_thresh})',
                unit='tile')):
            out_file = out_dir/f'{tile}.npy'
            if out_file.exists(): n_total+=1; continue

            if use_zip:
                with zf_h.open(pts_path) as fh: xyz=np.load(io.BytesIO(fh.read()))
            else: xyz=np.load(pts_path)
            N=len(xyz); n_total+=N

            if N>max_pts:
                sub_idx=rng.choice(N,max_pts,replace=False); sub=xyz[sub_idx]
            else: sub_idx=np.arange(N); sub=xyz

            f6=geo_features(sub)
            xn=sub.copy(); xn[:,:2]-=xn[:,:2].mean(0)
            xn[:,2]-=float(np.median(xn[:,2])); xn/=(np.abs(xn).max()+1e-6)
            pts9=np.concatenate([xn,f6],axis=1).astype(np.float32)
            del f6,xn

            tensor=torch.from_numpy(pts9).unsqueeze(0).to(DEVICE)
            del pts9
            with torch.no_grad():
                with torch.cuda.amp.autocast():
                    logits=base(tensor)
                probs=F.softmax(logits[0],-1)[:,1].cpu().numpy()
            del tensor,logits; torch.cuda.empty_cache()

            if N>max_pts:
                from scipy.spatial import cKDTree
                _,ni=cKDTree(sub).query(xyz,k=1,workers=-1)
                all_probs=probs[ni].astype(np.float32)
            else: all_probs=probs.astype(np.float32)
            del probs,sub

            conf=np.abs(all_probs-0.5)*2.
            model_lbl=(all_probs>=0.5).astype(np.int8)
            del all_probs

            cp=cons_dir/f'{tile}.npy'
            baseline=np.load(str(cp)).astype(np.int8) if cp.exists() else model_lbl.copy()
            refined=baseline.copy()
            hc=conf>=conf_thresh
            refined[hc]=model_lbl[hc]; n_repl+=int(hc.sum())
            del xyz,conf,model_lbl,baseline,hc
            np.save(str(out_file),refined.astype(np.int8))
            del refined
            if ti%500==0 and ti>0: gc.collect()
    finally:
        if zf_h: zf_h.close()

    gc.collect(); torch.cuda.empty_cache()
    pct=100*n_repl/max(n_total,1)
    print(f'  Replaced {pct:.1f}% of labels (conf>={conf_thresh})')
    print(f'  (High confidence = very pure labels)')
    return str(out_dir)


print('Generating Round 3 pseudo-labels...')
PSEUDO_DIR_R3 = generate_round3_pseudolabels(
    model, CFG, conf_thresh=CFG['pseudo_conf'])
print(f'Pseudo-labels saved: {PSEUDO_DIR_R3}')


Consensus labels: train=0, val=0
Building consensus labels...


Consensus[val]: 100%|██████████| 1216/1216 [00:02<00:00, 516.17it/s]


Generating Round 3 pseudo-labels...


Round3 pseudo-labels (conf>=0.95):   0%|          | 0/4922 [00:00<?, ?tile/s]/tmp/ipykernel_55/465830544.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Round3 pseudo-labels (conf>=0.95): 100%|██████████| 4922/4922 [08:49<00:00,  9.30tile/s]

  Replaced 2.7% of labels (conf>=0.95)
  (High confidence = very pure labels)
Pseudo-labels saved: /kaggle/working/pseudo_labels/Round3


In [7]:
# Cell 7 — Score all training tiles, identify hardest examples
#
# Hard Example Mining (HEM):
# We run the model on every training tile and compute per-tile accuracy.
# The hardest 40% of tiles (lowest accuracy) get extra training attention.
# This forces the model to focus on what it gets WRONG rather than
# endlessly re-learning what it already knows.
#
# Why this works:
# A model at 93.65% has already mastered 93.65% of examples.
# Standard training keeps showing it those easy examples every epoch.
# HEM removes easy examples and concentrates gradient signal on the
# genuinely ambiguous cases (kuccha walls, haystacks, dense canopy edges).

print('Scoring all training tiles for Hard Example Mining...')
print('(This takes ~5 min — runs model on every tile once)')

base = model.module if hasattr(model,'module') else model
base.eval()

# Score dataset using PSEUDO labels (most accurate labels available)
score_ds = GroundDataset(CFG, 'train', augment=False,
                          pseudo_dir=PSEUDO_DIR_R3)
score_ld = DataLoader(score_ds, batch_size=CFG['batch_size'],
                       shuffle=False, drop_last=False,
                       num_workers=CFG['num_workers'],
                       pin_memory=True)

tile_accs = []   # (accuracy, tile_name) for each tile
tile_idx  = 0

with torch.no_grad():
    for pts, lbs in tqdm(score_ld, desc='Scoring tiles', ncols=88):
        pts = pts.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits = base(pts)              # (B, N, 2)
        preds = logits.argmax(-1)           # (B, N)

        # Per-tile accuracy within this batch
        for b in range(pts.shape[0]):
            if tile_idx >= len(score_ds._tiles): break
            t_acc = (preds[b].cpu() == lbs[b]).float().mean().item()
            tile_accs.append((t_acc, score_ds._tiles[tile_idx]))
            tile_idx += 1

torch.cuda.empty_cache(); gc.collect()

# Sort by accuracy ascending (worst first)
tile_accs.sort(key=lambda x: x[0])

n_hard   = int(len(tile_accs) * CFG['hem_hard_frac'])
HARD_TILES   = [t for _, t in tile_accs[:n_hard]]       # worst 40%
MEDIUM_TILES = [t for _, t in tile_accs[n_hard:]]       # best 60%

# Stats
hard_accs   = [a for a,_ in tile_accs[:n_hard]]
medium_accs = [a for a,_ in tile_accs[n_hard:]]

print(f'\nTotal tiles scored : {len(tile_accs)}')
print(f'Hard tiles (worst {CFG["hem_hard_frac"]*100:.0f}%): {n_hard}')
print(f'  Mean accuracy    : {np.mean(hard_accs)*100:.2f}%')
print(f'  Min  accuracy    : {np.min(hard_accs)*100:.2f}%')
print(f'Easy tiles (best 60%): {len(MEDIUM_TILES)}')
print(f'  Mean accuracy    : {np.mean(medium_accs)*100:.2f}%')
print(f'\nWorst 5 tiles:')
for acc,tile in tile_accs[:5]:
    print(f'  {tile}  acc={acc*100:.1f}%')


Scoring all training tiles for Hard Example Mining...
(This takes ~5 min — runs model on every tile once)
  [train] 4922 tiles | lbl=pseudo | mixup=False


Scoring tiles:   0%|                                            | 0/308 [00:00<?, ?it/s]/tmp/ipykernel_55/3625472414.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Scoring tiles: 100%|██████████████████████████████████| 308/308 [00:57<00:00,  5.40it/s]


Total tiles scored : 4922
Hard tiles (worst 40%): 1968
  Mean accuracy    : 80.15%
  Min  accuracy    : 47.12%
Easy tiles (best 60%): 2954
  Mean accuracy    : 90.65%

Worst 5 tiles:
  tile_000761  acc=47.1%
  tile_004011  acc=50.1%
  tile_004332  acc=55.3%
  tile_005776  acc=56.5%
  tile_001644  acc=57.4%


In [8]:
# Cell 8 — Fine-tuning in two sub-phases
#
# Sub-phase A: Fine-tune on Round 3 pseudo-labels, full dataset + Mixup
#   Epochs: 15, LR: 3e-4 (low — model already trained)
#   Expected: ~94.5-95.5%
#
# Sub-phase B: Hard Example Mining — train ONLY on worst 40% of tiles
#   Epochs: 12, LR: 1e-4 (very low — targeted refinement)
#   Expected: ~96-97%+
#
# Sub-phase C: SWA sweep of 5 epochs (accumulate averaged weights)
#   + CPU BN update (avoids OOM)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.optim.swa_utils import AveragedModel, SWALR


class FocalLoss(nn.Module):
    def __init__(self,a,g,s):
        super().__init__(); self.a=a; self.g=g; self.s=s
    def forward(self,logits,targets):
        nc=logits.size(-1)
        soft=torch.zeros_like(logits).scatter_(1,targets.unsqueeze(1),1.)
        soft=soft*(1-self.s)+self.s/nc
        ce=-(soft*torch.log_softmax(logits,-1)).sum(-1)
        pt=torch.exp(-ce)
        at=torch.where(targets==1,
            targets.new_full(targets.shape,self.a,dtype=torch.float32),
            targets.new_full(targets.shape,1-self.a,dtype=torch.float32))
        return (at*(1-pt)**self.g*ce).mean()


def _metrics(preds,labels):
    acc=(preds==labels).float().mean().item()
    tp=((preds==1)&(labels==1)).sum().item()
    fp=((preds==1)&(labels==0)).sum().item()
    fn=((preds==0)&(labels==1)).sum().item()
    p=tp/(tp+fp+1e-9); r=tp/(tp+fn+1e-9)
    return acc,2*p*r/(p+r+1e-9)


def _autocast():
    try: return torch.amp.autocast('cuda',enabled=CFG['use_amp'])
    except: return torch.cuda.amp.autocast(enabled=CFG['use_amp'])

def _make_scaler():
    try: return torch.amp.GradScaler('cuda',enabled=CFG['use_amp'])
    except: return torch.cuda.amp.GradScaler(enabled=CFG['use_amp'])


def run_epoch(model, loader, opt, scaler, crit, base,
              train=True, desc=''):
    model.train() if train else model.eval()
    total_loss=0.; all_p=[]; all_l=[]

    ctx = _autocast() if train else torch.no_grad()
    with (ctx if not train else contextlib.nullcontext()):
        for pts,lbs in tqdm(loader,desc=desc,leave=False,ncols=90):
            pts=pts.to(DEVICE,non_blocking=True)
            lbs=lbs.to(DEVICE,non_blocking=True)
            if train:
                opt.zero_grad()
                with _autocast():
                    logits=model(pts)
                    loss=crit(logits.view(-1,2),lbs.view(-1))
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(base.parameters(),CFG['grad_clip'])
                scaler.step(opt); scaler.update()
            else:
                with _autocast():
                    logits=model(pts)
                    loss=crit(logits.view(-1,2),lbs.view(-1))
            total_loss+=loss.item()
            with torch.no_grad():
                all_p.append(logits.detach().argmax(-1).view(-1).cpu())
                all_l.append(lbs.view(-1).cpu())

    acc,f1=_metrics(torch.cat(all_p),torch.cat(all_l))
    return total_loss/len(loader),acc,f1


import contextlib

crit    = FocalLoss(CFG['focal_alpha'],CFG['focal_gamma'],CFG['label_smooth'])
base    = model.module if hasattr(model,'module') else model
va_ld   = make_loader(CFG,'val',augment=False)
ALL_H   = []; best_acc = 0.0

Path(CFG['logs_dir']).mkdir(parents=True,exist_ok=True)


# ══ SUB-PHASE A: Fine-tune on Round 3 pseudo-labels + Mixup ══════════════════
print('\n' + '='*60)
print('  SUB-PHASE A: Fine-tune (Round3 labels + Mixup)')
print(f'  Epochs: {CFG["finetune_epochs"]}  LR: {CFG["finetune_lr"]}')
print('='*60)

tr_ld_A = make_loader(CFG,'train',augment=True,pseudo_dir=PSEUDO_DIR_R3,
                       balanced=True,mixup=True)

opt_A = torch.optim.AdamW(base.parameters(),
                            lr=CFG['finetune_lr'],
                            weight_decay=CFG['weight_decay'])

# Cosine LR from finetune_lr → finetune_lr*0.01
sched_A = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_A, T_max=CFG['finetune_epochs'], eta_min=CFG['finetune_lr']*0.01)

scaler = _make_scaler()

for ep in range(1, CFG['finetune_epochs']+1):
    if (time.time()-SESSION_START)/3600 >= SESSION_BUDGET_H:
        print('Budget reached'); break

    tl,ta,tf1=run_epoch(model,tr_ld_A,opt_A,scaler,crit,base,
                         train=True,desc=f'A Ep{ep:2d} train')
    sched_A.step()
    cur_lr=opt_A.param_groups[0]['lr']

    va,vf1=float('nan'),float('nan')
    if ep%CFG['val_every']==0 or ep==CFG['finetune_epochs']:
        _,va,vf1=run_epoch(model,va_ld,None,None,crit,base,
                            train=False,desc=f'A Ep{ep:2d} val')
        if va>best_acc:
            best_acc=va
            torch.save(base.state_dict(),CFG['model_save'])

    star=' BEST' if (not math.isnan(va) and va==best_acc) else ''
    print(f'A Ep{ep:2d} lr={cur_lr:.5f} tl={tl:.4f} ta={ta:.4f} '
          f'| va={va:.4f} vf1={vf1:.4f} best={best_acc:.4f}{star}')
    ALL_H.append(dict(phase='A',ep=ep,tl=tl,ta=ta,va=va,vf1=vf1))
    torch.save(dict(epoch=ep,phase='A',model=base.state_dict(),
                    best_acc=best_acc),CFG['ckpt'])


# ══ SUB-PHASE B: Hard Example Mining ════════════════════════════════════════
print('\n' + '='*60)
print('  SUB-PHASE B: Hard Example Mining')
print(f'  Training on worst {CFG["hem_hard_frac"]*100:.0f}% of tiles')
print(f'  Epochs: {CFG["hem_epochs"]}  LR: {CFG["hem_lr"]}')
print('='*60)

# Reload best weights before HEM
base.load_state_dict(torch.load(CFG['model_save'],map_location=DEVICE))

tr_ld_B = make_loader(CFG,'train',augment=True,pseudo_dir=PSEUDO_DIR_R3,
                       balanced=True,mixup=False,tile_subset=HARD_TILES)

opt_B = torch.optim.AdamW(base.parameters(),
                            lr=CFG['hem_lr'],
                            weight_decay=CFG['weight_decay'])
sched_B = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_B, T_max=CFG['hem_epochs'], eta_min=CFG['hem_lr']*0.01)

for ep in range(1, CFG['hem_epochs']+1):
    if (time.time()-SESSION_START)/3600 >= SESSION_BUDGET_H:
        print('Budget reached'); break

    tl,ta,tf1=run_epoch(model,tr_ld_B,opt_B,scaler,crit,base,
                         train=True,desc=f'B Ep{ep:2d} train')
    sched_B.step()
    cur_lr=opt_B.param_groups[0]['lr']

    va,vf1=float('nan'),float('nan')
    if ep%CFG['val_every']==0 or ep==CFG['hem_epochs']:
        _,va,vf1=run_epoch(model,va_ld,None,None,crit,base,
                            train=False,desc=f'B Ep{ep:2d} val')
        if va>best_acc:
            best_acc=va
            torch.save(base.state_dict(),CFG['model_save'])

    star=' BEST' if (not math.isnan(va) and va==best_acc) else ''
    print(f'B Ep{ep:2d} lr={cur_lr:.5f} tl={tl:.4f} ta={ta:.4f} '
          f'| va={va:.4f} vf1={vf1:.4f} best={best_acc:.4f}{star}')
    ALL_H.append(dict(phase='B',ep=ep,tl=tl,ta=ta,va=va,vf1=vf1))


# ══ SUB-PHASE C: SWA (5 epochs) — CPU BN update to avoid OOM ════════════════
print('\n' + '='*60)
print('  SUB-PHASE C: SWA (5 epochs)')
print('='*60)

base.load_state_dict(torch.load(CFG['model_save'],map_location=DEVICE))
swa_m = AveragedModel(base)
opt_C = torch.optim.AdamW(base.parameters(),lr=CFG['swa_lr'],
                            weight_decay=CFG['weight_decay'])
swa_s = SWALR(opt_C,swa_lr=CFG['swa_lr'])
tr_ld_C = make_loader(CFG,'train',augment=True,pseudo_dir=PSEUDO_DIR_R3,
                       balanced=True,mixup=False)

for ep in range(1,6):
    if (time.time()-SESSION_START)/3600 >= SESSION_BUDGET_H: break
    tl,ta,tf1=run_epoch(model,tr_ld_C,opt_C,scaler,crit,base,
                         train=True,desc=f'C Ep{ep} swa')
    swa_m.update_parameters(base); swa_s.step()
    print(f'C Ep{ep} tl={tl:.4f} ta={ta:.4f} (SWA accumulating)')

# CPU BN update — the fix that prevents the OOM crash
print('SWA BN update on CPU (avoids GPU OOM)...')
torch.cuda.empty_cache(); gc.collect()
swa_cpu = swa_m.cpu()
_bn_ds  = GroundDataset(CFG,'train',augment=False,pseudo_dir=PSEUDO_DIR_R3)
import random as _rand
_bn_ds._tiles = _rand.sample(_bn_ds._tiles, min(400,len(_bn_ds._tiles)))
_bn_ld  = DataLoader(_bn_ds,batch_size=4,shuffle=False,
                      num_workers=0,pin_memory=False)
torch.optim.swa_utils.update_bn(_bn_ld,swa_cpu,device=torch.device('cpu'))
torch.save(swa_cpu.module.state_dict(),CFG['swa_save'])
del swa_cpu,_bn_ld,_bn_ds; gc.collect()
print(f'SWA saved: {CFG["swa_save"]}')

print(f'\nFine-tuning complete. Best val accuracy: {best_acc*100:.2f}%')


  [val] 1216 tiles | lbl=consensus | mixup=False

  SUB-PHASE A: Fine-tune (Round3 labels + Mixup)
  Epochs: 15  LR: 0.0003
  [train] 4922 tiles | lbl=pseudo | mixup=True


A Ep 1 lr=0.00030 tl=0.0239 ta=0.8587 | va=nan vf1=nan best=0.0000


A Ep 2 lr=0.00029 tl=0.0209 ta=0.8655 | va=0.9121 vf1=0.8776 best=0.9121 BEST


A Ep 3 lr=0.00027 tl=0.0204 ta=0.8697 | va=nan vf1=nan best=0.9121


A Ep 4 lr=0.00025 tl=0.0202 ta=0.8704 | va=0.9158 vf1=0.8815 best=0.9158 BEST


...

C Ep1 tl=0.0145 ta=0.9120 (SWA accumulating)


C Ep2 tl=0.0142 ta=0.9132 (SWA accumulating)


C Ep3 tl=0.0142 ta=0.9134 (SWA accumulating)


C Ep4 tl=0.0142 ta=0.9134 (SWA accumulating)


C Ep5 tl=0.0141 ta=0.9135 (SWA accumulating)
SWA BN update on CPU (avoids GPU OOM)...
  [train] 4922 tiles | lbl=pseudo | mixup=False
SWA saved: /kaggle/working/logs/swa_model.pth

Fine-tuning complete. Best val accuracy: 92.15%


In [9]:
# Cell 9 — TTA Threshold Sweep + Package
#
# Test-Time Augmentation (TTA) at validation:
# Run the model N times with different random subsamples of each tile.
# Average the predicted probabilities before thresholding.
# This reduces variance and typically adds +0.3-0.8pp accuracy — for free.
#
# We sweep both models (best_epoch and SWA) with and without TTA.
# Pick the winner.

def sweep_with_tta(model_path, cfg, device, label='', tta_passes=8):
    """Threshold sweep with test-time augmentation."""
    if not Path(model_path).exists(): return None

    m = DTMPointNet2(IN_FEAT).to(device)
    m.load_state_dict(torch.load(model_path,map_location=device))
    m.eval()

    # Build val dataset without cache (so augmentation affects features too)
    va_ds = GroundDataset(cfg,'val',augment=False)
    rng   = np.random.default_rng(42)

    all_probs = []
    all_labels= []

    print(f'TTA sweep [{label}]: {tta_passes} passes x {len(va_ds)} tiles...')

    for tile_idx in tqdm(range(len(va_ds)), desc=f'TTA {label}',
                          ncols=88, leave=False):
        # Load tile directly
        pts_base, lbl = va_ds[tile_idx]
        N = pts_base.shape[0]
        avg_p = np.zeros(N, dtype=np.float32)

        for _ in range(tta_passes):
            # Different random subsample each pass
            if 'train' not in va_ds.split:  # always val
                perm  = rng.choice(N, N, replace=False)
                pts_t = pts_base[perm].unsqueeze(0).to(device)
                with torch.no_grad():
                    with torch.cuda.amp.autocast():
                        logits = m(pts_t)
                    p = F.softmax(logits[0],-1)[:,1].cpu().numpy()
                # Map back to original order
                avg_p[perm] += p

        avg_p /= tta_passes
        all_probs.append(avg_p)
        all_labels.append(lbl.numpy())

    probs = np.concatenate(all_probs)
    labs  = np.concatenate(all_labels)

    best_thr,best_f1,best_acc = 0.5,0.,0.; curve=[]
    for thr in np.arange(0.10,0.90,0.01):
        p=(probs>=thr).astype(np.int64)
        tp=((p==1)&(labs==1)).sum(); fp=((p==1)&(labs==0)).sum()
        fn=((p==0)&(labs==1)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9)
        f1=2*pr*rc/(pr+rc+1e-9); acc=(p==labs).mean()
        curve.append((float(thr),float(f1),float(acc)))
        if f1>best_f1: best_f1,best_thr,best_acc=f1,float(thr),float(acc)

    return dict(model_path=model_path,threshold=best_thr,
                val_accuracy=best_acc,val_f1=best_f1,curve=curve,
                tta_passes=tta_passes)


results={}
TTA = CFG['val_tta_passes']

for lbl,path in [('best_epoch+TTA', CFG['model_save']),
                  ('swa+TTA',        CFG['swa_save'])]:
    r = sweep_with_tta(path,CFG,DEVICE,lbl,tta_passes=TTA)
    if r:
        results[lbl]=r
        print(f'[{lbl}]  thr={r["threshold"]:.2f}  '
              f'acc={r["val_accuracy"]*100:.2f}%  f1={r["val_f1"]:.4f}')

if not results:
    print('No models found — run Cell 8 first'); raise SystemExit()

winner = max(results,key=lambda k:results[k]['val_accuracy'])
best   = results[winner]
print(f'\nWinner: {winner}  acc={best["val_accuracy"]*100:.2f}%')

# Plot
c=best['curve']
fig,ax=plt.subplots(figsize=(8,4))
ax.plot([x[0] for x in c],[x[1] for x in c],label='F1')
ax.plot([x[0] for x in c],[x[2] for x in c],label='Accuracy')
ax.axvline(best['threshold'],ls='--',c='red',label=f'thr={best["threshold"]:.2f}')
ax.axhline(0.95,ls=':',c='orange',alpha=0.7,label='95%')
ax.axhline(0.97,ls=':',c='green', alpha=0.7,label='97%')
ax.set(xlabel='Threshold',ylim=(0.85,1.0),title=f'Threshold sweep ({winner})')
ax.legend(); plt.tight_layout()
plt.savefig(str(Path(CFG['logs_dir'])/'threshold_curve.png'),dpi=120)
plt.close()

meta=dict(model=winner,model_path=best['model_path'],
          threshold=best['threshold'],val_accuracy=best['val_accuracy'],
          val_f1=best['val_f1'],tta_passes=TTA)
with open(CFG['threshold'],'w') as f: json.dump(meta,f,indent=2)

# Package
files=[best['model_path'],CFG['model_save'],CFG['swa_save'],
       CFG['threshold'],CFG['history'],CFG['curves'],
       str(Path(CFG['logs_dir'])/'threshold_curve.png')]

out_zip=Path('/kaggle/working/dtm_outputs_finetuned.zip')
with zipfile.ZipFile(str(out_zip),'w',zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        if Path(p).exists():
            zf.write(p,Path(p).name)
            print(f'  packed {Path(p).name} ({Path(p).stat().st_size/1e3:.0f} KB)')

print(f'\ndtm_outputs_finetuned.zip: {out_zip.stat().st_size/1e6:.1f} MB')
print(f'\nFINAL RESULT')
print(f'  Model      : {winner}')
print(f'  Threshold  : {best["threshold"]:.2f}')
print(f'  TTA passes : {TTA}')
print(f'  Accuracy   : {best["val_accuracy"]*100:.2f}%')
print(f'  F1 (ground): {best["val_f1"]:.4f}')
print(f'  Prev. best : 93.65%')
print(f'  Gain       : +{(best["val_accuracy"]-0.9365)*100:.2f}pp')
if best['val_accuracy']>=0.97: print('  TARGET 97%+: HIT')
elif best['val_accuracy']>=0.95: print('  TARGET 95%+: HIT (97% needs more rounds)')
else: print(f'  Gap to 97% : {(0.97-best["val_accuracy"])*100:.2f}pp')


  [val] 1216 tiles | lbl=consensus | mixup=False
TTA sweep [best_epoch+TTA]: 8 passes x 1216 tiles...


TTA best_epoch+TTA:   0%|                                      | 0/1216 [00:00<?, ?it/s]/tmp/ipykernel_55/2394271281.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[best_epoch+TTA]  thr=0.56  acc=93.97%  f1=0.9073
  [val] 1216 tiles | lbl=consensus | mixup=False
TTA sweep [swa+TTA]: 8 passes x 1216 tiles...


[swa+TTA]  thr=0.56  acc=93.62%  f1=0.9026

Winner: best_epoch+TTA  acc=93.97%
  packed best_model.pth (3577 KB)
  packed best_model.pth (3577 KB)


/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'best_model.pth'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


  packed swa_model.pth (3575 KB)
  packed threshold.json (0 KB)
  packed threshold_curve.png (46 KB)

dtm_outputs_finetuned.zip: 9.9 MB

FINAL RESULT
  Model      : best_epoch+TTA
  Threshold  : 0.56
  TTA passes : 8
  Accuracy   : 93.97%
  F1 (ground): 0.9073
  Prev. best : 93.65%
  Gain       : +0.32pp
  Gap to 97% : 3.03pp
